# HDB Resale Flat Valuation

This notebook estimates the fair market value of a Singapore HDB resale flat using three independent valuation approaches and blends them into a single point estimate and conservative range.

The subject property and comparable universe are configurable in the cell below — change the `SUBJECT_*` and `COMPARABLE_TOWNS` values to value any flat covered by the dataset. Inputs are validated against the loaded data, with errors that list valid values on typos.

## Data source

Live download of **Resale flat prices based on registration date from Jan-2017 onwards** from [data.gov.sg](https://data.gov.sg/datasets/d_8b84c4ee58e3cfc0ece0d773c8ca6abc/view).

**Caveats from HDB:**
- Approximate floor area includes any recess area purchased, space-adding items under upgrading programmes, roof terraces, and similar additions.
- The dataset excludes transactions that may not reflect the full market price (e.g., resale between relatives, resale of part shares).
- Resale prices are indicative only — actual sale prices depend on many factors not captured here.

In [ ]:
import sys
from pathlib import Path

# Make the repo-root `hdb_valuation` package importable regardless of where the
# notebook is launched (walk up from the working dir until we find the package).
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "hdb_valuation").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

# Dataset ID lives in the package so the notebook and dashboard can't disagree.
from hdb_valuation import DATASET_ID

print(DATASET_ID)

In [ ]:
import json
import requests

s = requests.Session()
s.headers.update({"referer": "https://colab.research.google.com"})

BASE_URL = "https://api-production.data.gov.sg"
url = f"{BASE_URL}/v2/public/api/datasets/{DATASET_ID}/metadata"
response = s.get(url)

data = response.json()["data"]
column_metadata = data.pop("columnMetadata", None)

print("Dataset:", data["name"])
print(f"Coverage: {data['coverageStart'][:10]} to {data['coverageEnd'][:10]}")
print(f"Last updated: {data['lastUpdatedAt'][:10]}")
print(f"Columns: {list(column_metadata['map'].values())}")

In [ ]:
import numpy as np
import pandas as pd

from hdb_valuation import fetch_raw_csv, engineer_features

# fetch_raw_csv touches the network; engineer_features is pure. Together they
# replace the old inline download_file + feature-engineering blocks.
df = engineer_features(fetch_raw_csv(DATASET_ID))
print(f"Loaded {len(df):,} rows")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from hdb_valuation import DJQC_COLORS
from hdb_valuation.plots import apply_style

# Same DJQC style the dashboard uses (DJQC_COLORS/sns used by the EDA cells below).
apply_style()

In [ ]:
from hdb_valuation import INCLUDED_MODELS, SQM_TO_SQFT

# ── User configuration ──────────────────────────────────────────────────────
# Subject property — the unit being valued.
SUBJECT_TOWN       = "QUEENSTOWN"           # must match HDB town names exactly
SUBJECT_FLAT_TYPE  = "4 ROOM"               # e.g. "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE"
SUBJECT_FLAT_MODEL = "Premium Apartment"    # e.g. "Model A", "Improved", "Premium Apartment"
SUBJECT_FLOOR      = 10                     # storey
SUBJECT_AREA_SQM   = 83                     # floor area in square metres
SUBJECT_LEASE_LEFT = 94.99                  # remaining lease in years
SUBJECT_STREET     = "DAWSON"               # substring matched against street_name (for the street-level highlight)

# Comparable universe for valuation. SUBJECT_TOWN is auto-added if missing.
# Use a single-element list (e.g. [SUBJECT_TOWN]) to restrict to the subject town only.
COMPARABLE_TOWNS = ["QUEENSTOWN", "BUKIT MERAH", "CENTRAL AREA"]

# ── Validate inputs against the loaded dataset ──────────────────────────────
SUBJECT_TOWN      = SUBJECT_TOWN.upper().strip()
SUBJECT_FLAT_TYPE = SUBJECT_FLAT_TYPE.upper().strip()
COMPARABLE_TOWNS  = [t.upper().strip() for t in COMPARABLE_TOWNS]

available_towns      = set(df["town"].unique())
available_models     = set(df["flat_model"].unique())
available_flat_types = set(df["flat_type"].unique())

unknown_towns = {SUBJECT_TOWN, *COMPARABLE_TOWNS} - available_towns
if unknown_towns:
    raise ValueError(
        f"Unknown town(s): {sorted(unknown_towns)}. "
        f"Available towns: {sorted(available_towns)}"
    )

if SUBJECT_FLAT_TYPE not in available_flat_types:
    raise ValueError(
        f"Unknown flat type {SUBJECT_FLAT_TYPE!r}. "
        f"Available flat types: {sorted(available_flat_types)}"
    )

if SUBJECT_FLAT_MODEL not in available_models:
    raise ValueError(
        f"Unknown flat model {SUBJECT_FLAT_MODEL!r}. "
        f"Available models: {sorted(available_models)}"
    )

if SUBJECT_FLOOR <= 0 or SUBJECT_AREA_SQM <= 0 or not (0 < SUBJECT_LEASE_LEFT <= 99):
    raise ValueError(
        f"SUBJECT_FLOOR (>0), SUBJECT_AREA_SQM (>0), SUBJECT_LEASE_LEFT (0-99) must be positive/in-range; "
        f"got {SUBJECT_FLOOR=}, {SUBJECT_AREA_SQM=}, {SUBJECT_LEASE_LEFT=}"
    )

if SUBJECT_TOWN not in COMPARABLE_TOWNS:
    print(f"Note: subject town {SUBJECT_TOWN!r} not in COMPARABLE_TOWNS; auto-adding.")
    COMPARABLE_TOWNS = [SUBJECT_TOWN, *COMPARABLE_TOWNS]

# Display-friendly label for plot/table titles, e.g. "4 ROOM" -> "4-Room".
SUBJECT_FLAT_TYPE_LABEL = SUBJECT_FLAT_TYPE.title().replace(" ", "-")

SELECTED_TOWNS = [SUBJECT_TOWN]  # used by exploratory cells below

In [ ]:
is_subject_type = df["flat_type"] == SUBJECT_FLAT_TYPE
is_selected_town = df["town"].isin(SELECTED_TOWNS)
is_recent = df["month"].dt.year >= 2024

df_subject_full      = df.loc[is_subject_type & is_selected_town]   # subject town, all years
df_all_recent        = df.loc[is_subject_type & is_recent]           # all towns, 2024+
df_subject_recent    = df_subject_full.loc[is_recent]                # subject town, 2024+


In [ ]:
plt.figure(figsize=(9, 5))

town_colors = {
    town: DJQC_COLORS[1] if town == SUBJECT_TOWN else "grey"
    for town in df_all_recent["town"].unique()
}

sns.boxenplot(
    data=df_all_recent, x="town", y="resale_price",
    hue="town", palette=town_colors, dodge=False,
)
plt.title(f"Flat Prices across Towns from 2024 onwards ({SUBJECT_FLAT_TYPE_LABEL} Flats)")
plt.xlabel("Town")
plt.ylabel("Resale Price")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
time_series = df_subject_full.groupby("month")["resale_price"].agg(
    ["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)]
)
time_series.columns = ["median", "Q1", "Q3"]
time_series["rolling_avg"] = time_series["median"].rolling(window=6).mean()

plt.figure(figsize=(9, 5))
plt.plot(time_series.index, time_series["median"], marker="o", label="Median", color=DJQC_COLORS[1], linewidth=2)
plt.plot(time_series.index, time_series["Q1"], marker="o", label="Q1", color="grey", linestyle=":", linewidth=1.5)
plt.plot(time_series.index, time_series["Q3"], marker="o", label="Q3", color="grey", linestyle=":", linewidth=1.5)
plt.plot(time_series.index, time_series["rolling_avg"], marker="o", label="6-Mo Rolling Avg", color="red", linestyle="--", linewidth=2)
plt.title(f"Resale Price of {SUBJECT_TOWN.title()} ({SUBJECT_FLAT_TYPE_LABEL} Flats)")
plt.xlabel("Month")
plt.ylabel("Resale Price")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
df_plot = df_subject_recent.dropna(subset=["remaining_lease_years"])

plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=df_plot, x="remaining_lease_years", y="resale_price",
    hue="flat_model", palette=DJQC_COLORS[1:],
    s=25, alpha=0.7,
)
plt.title(f"Flat Prices in {SUBJECT_TOWN.title()} ({SUBJECT_FLAT_TYPE_LABEL} Flats); 2024 onwards")
plt.xlabel("Remaining lease (Years)")
plt.ylabel("Resale price")
plt.legend(title="Flat Model", fontsize=8, title_fontsize=8, bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
from great_tables import *

In [ ]:
DISPLAY_COLS = [
    "month", "block", "street_name", "storey_range", "resale_price",
    "floor_area_sqm", "price_per_sqft", "flat_model", "remaining_lease_years",
]

transactions_2026 = (
    df.loc[
        (df["town"] == SUBJECT_TOWN)
        & (df["month"].dt.year == 2026)
        & (df["flat_type"] == SUBJECT_FLAT_TYPE)
        & (df["street_name"].str.contains(SUBJECT_STREET))
    ][DISPLAY_COLS]
    .sort_values("month", ascending=False)
    .reset_index(drop=True)
)


In [ ]:

fy2026_dawson = (
    GT(transactions_2026)
  .tab_header(title="FY 2026 Transactions", subtitle='Selected areas')
  .tab_stub(groupname_col="street_name", rowname_col="month")
  .cols_label( # Rename columns using cols_label()
      month="Transaction Month",
      block="Block",
      street_name="Street Name",
      storey_range="Storey Range",
      resale_price="Resale Price",
      floor_area_sqm="Area (sqm)",
      price_per_sqft="Price/sqft",
      flat_model="Flat Model",
      remaining_lease_years="Lease Left"
  )
  .data_color(
    columns=["resale_price"],
    domain = [400000, 1400000],
    palette=["lightgrey", 'lightblue', "green"]
  )
  .data_color(
    columns=["price_per_sqft"],
    domain = [500, 1400],
    palette=["lightgrey", 'lightblue', "green"]
  )
  .opt_table_font(font="Arial Narrow")
  .fmt_date(columns="month", date_style="month") # Update column name in fmt_date()
  .fmt_number(columns=["resale_price", "price_per_sqft"], compact=True) # Update column names in fmt_number()
  .fmt_number(columns=["remaining_lease_years"], decimals=2) # Update column names in fmt_number()
  .cols_align(align="left")
)

fy2026_dawson

## Fair Value Analysis

The cells below run three independent valuation approaches against the **subject property** and **comparable universe** defined in the configuration cell at the top of the notebook:

| Input | Source |
|---|---|
| Subject town, flat type, flat model, floor, area, lease, street | `SUBJECT_*` configuration variables |
| Comparable towns | `COMPARABLE_TOWNS` |
| Comparable flat models | `INCLUDED_MODELS` (set in the cell directly below) |

The comparable universe is restricted to **2024+ transactions** so the valuation reflects current market conditions rather than the full historical sweep.

### The three approaches

1. **Historical Comparables** — pick the 20 transactions most similar to the subject (weighted distance over floor, lease, area), then adjust each comp's price to the subject's exact attributes. Take the median.
2. **Regression with time trend** — OLS on the comparable universe with `floor_area_sqm`, `storey_mid`, `remaining_lease_years`, `months_since_2024`, plus one-hot `town` and `flat_model`. The time feature captures recent appreciation.
3. **Regression without time trend** — same model, time feature dropped. Treats all 2024+ transactions as equally current; useful as a sensitivity check against (2).

The three point estimates are averaged into a **blended fair value**, and the conservative range spans the widest of the three confidence intervals.

In [ ]:
from hdb_valuation import Subject, build_universe

# ── Subject property ─────────────────────────────────────────────────────────
# Values come from the user-configuration cell near the top of the notebook.
subject = Subject(
    town=SUBJECT_TOWN,
    flat_type=SUBJECT_FLAT_TYPE,
    flat_model=SUBJECT_FLAT_MODEL,
    floor=SUBJECT_FLOOR,
    area_sqm=SUBJECT_AREA_SQM,
    lease_left=SUBJECT_LEASE_LEFT,
    street=SUBJECT_STREET,
)

# ── Comparable universe ──────────────────────────────────────────────────────
# INCLUDED_MODELS is defined in the package (imported in the config cell above).
df_comp, df_comp_recent = build_universe(
    df, SUBJECT_TOWN, SUBJECT_FLAT_TYPE, COMPARABLE_TOWNS, INCLUDED_MODELS
)

print(f"Comparable towns: {COMPARABLE_TOWNS}")
print(f"Full comparable universe (2017-2026):  {len(df_comp):,} txns")
print(f"Recent comparables (2024+):            {len(df_comp_recent):,} txns")
print(f"\nBy town:\n{df_comp_recent['town'].value_counts().to_string()}")
print(f"\nBy model:\n{df_comp_recent['flat_model'].value_counts().to_string()}")

### Approach 1 — Historical Comparables

Pick the 20 transactions most similar to the subject (weighted distance on floor, lease, and area), then adjust each comparable's sale price to the subject's exact attributes. The point estimate is the median adjusted price across those 20 comps.

Adjustment rates are univariate OLS slopes fit on the comparable universe:

- **Floor-level premium** — $/floor
- **Lease premium** — $/year of remaining lease
- **Area adjustment** — $/sqm difference
- **Time adjustment** — $/month of market appreciation between the comp's transaction date and the valuation month

In [ ]:
from hdb_valuation import fit_factors

# Univariate OLS slopes ($ per unit) used to adjust each comparable.
factors = fit_factors(df_comp_recent)

for key, label in [
    ("floor-level", "Floor-level ($/floor)"),
    ("lease",       "Lease ($/year)"),
    ("area",        "Area ($/sqm)"),
    ("time",        "Time ($/month)"),
]:
    print(f"{label:<25s}  ${factors[key]:>10,.0f}")

In [ ]:
from datetime import datetime

from hdb_valuation import months_since_2024, valuation_comparables

# Valuation month = current month (Jan 2024 = 1). The dashboard uses the same
# call, so the two front-ends value against the same point in time.
VALUATION_MONTH = months_since_2024(datetime.now())

comp = valuation_comparables(df_comp_recent, subject, factors, VALUATION_MONTH)
top_comps = comp["top_comps"]

print("Top 20 comparables - adjusted price statistics:")
for stat, val in [
    ("Mean", top_comps["adjusted_price"].mean()),
    ("Median", top_comps["adjusted_price"].median()),
    ("Std", top_comps["adjusted_price"].std()),
    ("Range", None),
]:
    if stat == "Range":
        print(f"  {stat:<8s} ${top_comps['adjusted_price'].min():>12,.0f} - ${top_comps['adjusted_price'].max():>12,.0f}")
    else:
        print(f"  {stat:<8s} ${val:>12,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Adjusted price distribution
axes[0].hist(top_comps["adjusted_price"], bins=12, color=DJQC_COLORS[1], edgecolor="white", alpha=0.85)
axes[0].axvline(
    top_comps["adjusted_price"].median(), color=DJQC_COLORS[2], linestyle="--", linewidth=2,
    label=f"Median: ${top_comps['adjusted_price'].median():,.0f}",
)
axes[0].set_title("Adjusted Price Distribution (Top 20 Comps)")
axes[0].set_xlabel("Adjusted Price ($)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=8)

# 2. Raw vs adjusted
axes[1].scatter(top_comps["resale_price"], top_comps["adjusted_price"], color=DJQC_COLORS[1], s=30, alpha=0.7)
lims = [
    min(top_comps["resale_price"].min(), top_comps["adjusted_price"].min()) - 20_000,
    max(top_comps["resale_price"].max(), top_comps["adjusted_price"].max()) + 20_000,
]
axes[1].plot(lims, lims, "k--", alpha=0.3, label="No adjustment")
axes[1].set_title("Raw vs Adjusted Price")
axes[1].set_xlabel("Raw Transaction Price ($)")
axes[1].set_ylabel("Adjusted Price ($)")
axes[1].legend(fontsize=8)

# 3. Adjustment breakdown
adj_cols = ["adj_floor-level", "adj_lease", "adj_area", "adj_time"]
adj_means = top_comps[adj_cols].mean()
axes[2].barh(
    ["Floor", "Lease", "Area", "Time"], adj_means.values,
    color=[DJQC_COLORS[2], DJQC_COLORS[3], DJQC_COLORS[4], DJQC_COLORS[5]],
)
axes[2].axvline(0, color="black", linewidth=0.8)
axes[2].set_title("Mean Adjustment Breakdown")
axes[2].set_xlabel("Adjustment ($)")

plt.tight_layout()
plt.show()

In [ ]:
# ── Comparables table ───────────────────────────────────────────────────────

comp_display = top_comps[[
    'month', 'town', 'block', 'street_name', 'storey_range',
    'floor_area_sqm', 'remaining_lease_years', 'flat_model',
    'resale_price', 'total_adjustment', 'adjusted_price'
]].sort_values('month', ascending=False).reset_index(drop=True)

comp_gt = (
    GT(comp_display)
    .tab_header(
        title="Approach 1 — Comparable Transactions (Adjusted)",
        subtitle="Top 20 closest comps · Model A & Premium Apartment · 2024 onwards"
    )
    .cols_label(
        month="Month", town="Town", block="Blk", street_name="Street",
        storey_range="Floor", floor_area_sqm="Area (sqm)",
        remaining_lease_years="Lease Left", flat_model="Model",
        resale_price="Raw Price", total_adjustment="Adj.",
        adjusted_price="Adjusted Price"
    )
    .data_color(
        columns=["adjusted_price"],
        domain=[comp_display['adjusted_price'].min(), comp_display['adjusted_price'].max()],
        palette=["lightgrey", "lightblue", "green"]
    )
    .opt_table_font(font="Arial Narrow")
    # .fmt_date(columns="month", date_style="month")
    .fmt_number(columns=["resale_price", "total_adjustment", "adjusted_price"], compact=True)
    .fmt_number(columns=["remaining_lease_years"], decimals=1)
    .cols_align(align="left")
)

comp_gt

In [ ]:
comp_estimate = comp["estimate"]

print("APPROACH 1 - HISTORICAL COMPARABLES")
print(f"  Fair value (median):   ${comp['estimate']:>12,.0f}")
print(f"  IQR:                   ${comp['q25']:>10,.0f} - ${comp['q75']:>10,.0f}")
print(f"  Price per sqft:        ${comp['estimate'] / subject.sqft:>10,.0f}")

### Approach 2a — Regression Model (with time trend)

OLS regression trained on the recent (2024+) comparable universe.

**Features:**
- `floor_area_sqm`, `storey_mid`, `remaining_lease_years` — numeric
- `months_since_2024` — captures recent market appreciation
- `town`, `flat_model` — one-hot encoded

Numeric features are standardised before fitting (avoids matmul overflow when un-scaled values span many orders of magnitude).

In [ ]:
from hdb_valuation import (
    valuation_regression, NUM_COLS_WITH_TIME, NUM_COLS_NO_TIME, CAT_COLS,
)

# Fit + predict happen inside the package; the returned dict also carries the
# fitted model and predictions so the diagnostic cells below can inspect them.
reg_a = valuation_regression(df_comp_recent, subject, VALUATION_MONTH, with_time=True)

reg_estimate = reg_a["estimate"]
residual_std = reg_a["residual_std"]
reg_model    = reg_a["model"]
y_pred       = reg_a["y_pred"]
df_reg       = reg_a["df_reg"]

print(f"Approach 2a - OLS with time trend (n={reg_a['n']:,})")
print(f"  R²:  {reg_a['r2']:.4f}")
print(f"  MAE: ${reg_a['mae']:,.0f}")

In [ ]:
regressor = reg_model.named_steps["regressor"]
scaler = reg_model.named_steps["preprocessor"].named_transformers_["num"]
ohe = reg_model.named_steps["preprocessor"].named_transformers_["cat"]

num_coefs = regressor.coef_[:len(NUM_COLS_WITH_TIME)] / scaler.scale_
cat_names = list(ohe.get_feature_names_out(CAT_COLS))
cat_coefs = regressor.coef_[len(NUM_COLS_WITH_TIME):]

coef_df = pd.DataFrame({
    "Feature": NUM_COLS_WITH_TIME + cat_names,
    "Coefficient": list(num_coefs) + list(cat_coefs),
}).sort_values("Coefficient", key=abs, ascending=False)

print("Feature Coefficients (original units):")
print(coef_df.to_string(index=False, float_format="${:>12,.0f}".format))
print(f"\nIntercept: ${regressor.intercept_:,.0f}")

In [ ]:
residuals = df_reg["resale_price"] - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_pred, df_reg["resale_price"], s=5, alpha=0.3, color=DJQC_COLORS[1])
lims = [df_reg["resale_price"].min() - 20_000, df_reg["resale_price"].max() + 20_000]
axes[0].plot(lims, lims, "k--", alpha=0.4)
axes[0].set_title("Predicted vs Actual")
axes[0].set_xlabel("Predicted Price ($)")
axes[0].set_ylabel("Actual Price ($)")

axes[1].hist(residuals, bins=50, color=DJQC_COLORS[1], edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Residuals (std=${residuals.std():,.0f})")
axes[1].set_xlabel("Residual ($)")
axes[1].set_ylabel("Count")

axes[2].scatter(df_reg["months_since_2024"], residuals, s=5, alpha=0.3, color=DJQC_COLORS[1])
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_title("Residuals vs Time")
axes[2].set_xlabel("Months since Jan 2024")
axes[2].set_ylabel("Residual ($)")

plt.tight_layout()
plt.show()

In [ ]:
print("APPROACH 2a - REGRESSION (WITH TIME TREND)")
print(f"  Predicted fair value:  ${reg_estimate:>12,.0f}")
print(f"  68% range (±1 std):    ${reg_estimate - residual_std:>10,.0f} - ${reg_estimate + residual_std:>10,.0f}")
print(f"  95% range (±2 std):    ${reg_estimate - 2*residual_std:>10,.0f} - ${reg_estimate + 2*residual_std:>10,.0f}")
print(f"  Price per sqft:        ${reg_estimate / subject.sqft:>10,.0f}")

### Approach 2b — Regression Model (without time trend)

Same model and universe as 2a, but `months_since_2024` is dropped. Treats all 2024+ transactions as equally current — a snapshot estimate that ignores the recent trend. Useful as a sensitivity check against Approach 2a's time-adjusted estimate: a large gap between the two is a measure of how much of the predicted value is being driven by extrapolated appreciation.

In [ ]:
reg_b = valuation_regression(df_comp_recent, subject, VALUATION_MONTH, with_time=False)

reg_b_estimate = reg_b["estimate"]
residual_b_std = reg_b["residual_std"]
reg_model_b    = reg_b["model"]
y_pred_b       = reg_b["y_pred"]

print(f"Approach 2b - OLS without time trend (n={reg_b['n']:,})")
print(f"  R²:  {reg_b['r2']:.4f}")
print(f"  MAE: ${reg_b['mae']:,.0f}")
print(f"\n  vs Approach 2a: R² delta = {reg_a['r2'] - reg_b['r2']:+.4f}, MAE delta = ${reg_a['mae'] - reg_b['mae']:+,.0f}")

In [ ]:
regressor_b = reg_model_b.named_steps["regressor"]
scaler_b = reg_model_b.named_steps["preprocessor"].named_transformers_["num"]
ohe_b = reg_model_b.named_steps["preprocessor"].named_transformers_["cat"]

num_coefs_b = regressor_b.coef_[:len(NUM_COLS_NO_TIME)] / scaler_b.scale_
cat_names_b = list(ohe_b.get_feature_names_out(CAT_COLS))
cat_coefs_b = regressor_b.coef_[len(NUM_COLS_NO_TIME):]

coef_df_b = pd.DataFrame({
    "Feature": NUM_COLS_NO_TIME + cat_names_b,
    "Coefficient": list(num_coefs_b) + list(cat_coefs_b),
}).sort_values("Coefficient", key=abs, ascending=False)

print("Feature Coefficients - Approach 2b (no time trend):")
print(coef_df_b.to_string(index=False, float_format="${:>12,.0f}".format))
print(f"\nIntercept: ${regressor_b.intercept_:,.0f}")

print(f"\nAPPROACH 2b - REGRESSION (NO TIME TREND)")
print(f"  Predicted fair value:  ${reg_b_estimate:>12,.0f}")
print(f"  68% range (±1 std):    ${reg_b_estimate - residual_b_std:>10,.0f} - ${reg_b_estimate + residual_b_std:>10,.0f}")
print(f"  95% range (±2 std):    ${reg_b_estimate - 2*residual_b_std:>10,.0f} - ${reg_b_estimate + 2*residual_b_std:>10,.0f}")
print(f"  Price per sqft:        ${reg_b_estimate / subject.sqft:>10,.0f}")

### Combined Valuation Summary

The three approaches give independent estimates of the same quantity. The blended point estimate is their **simple average**; the conservative range spans the widest of the three confidence intervals (IQR for Approach 1, ±1 residual std for Approaches 2a / 2b).

A tight cluster across the three approaches indicates convergence and higher confidence in the estimate; a wide spread suggests the result is sensitive to modelling choices and the range should be taken seriously.

In [ ]:
from hdb_valuation import blend

blended = blend(comp, reg_a, reg_b, subject)
blended_estimate = blended["blended"]
range_low  = blended["low"]
range_high = blended["high"]
psf = blended["psf"]

comp_estimate = comp["estimate"]
comp_q25 = comp["q25"]
comp_q75 = comp["q75"]

val_label = datetime.now().strftime("%b %Y")

summary_data = pd.DataFrame({
    "Approach": [
        "1. Historical Comparables",
        "2a. Regression (with time)",
        "2b. Regression (no time)",
        "Blended Estimate",
    ],
    "Point Estimate ($)": [comp_estimate, reg_estimate, reg_b_estimate, blended_estimate],
    "Low ($)": [comp_q25, reg_estimate - residual_std, reg_b_estimate - residual_b_std, range_low],
    "High ($)": [comp_q75, reg_estimate + residual_std, reg_b_estimate + residual_b_std, range_high],
    "Price/sqft ($)": [
        comp_estimate / subject.sqft,
        reg_estimate / subject.sqft,
        reg_b_estimate / subject.sqft,
        psf,
    ],
})

summary_gt = (
    GT(summary_data)
    .tab_header(
        title=f"Fair Value Summary - {subject.street.title()} Road, {SUBJECT_FLAT_TYPE_LABEL} {subject.flat_model}",
        subtitle=f"{subject.area_sqm} sqm | Floor {subject.floor} | {subject.lease_left} yrs lease | {val_label}",
    )
    .cols_label(**{
        "Approach": "Approach", "Point Estimate ($)": "Point Estimate",
        "Low ($)": "Low", "High ($)": "High", "Price/sqft ($)": "$/sqft",
    })
    .fmt_currency(columns=["Point Estimate ($)", "Low ($)", "High ($)"], currency="USD", decimals=0)
    .fmt_number(columns=["Price/sqft ($)"], decimals=0)
    .data_color(
        columns=["Point Estimate ($)"],
        domain=[summary_data["Point Estimate ($)"].min(), summary_data["Point Estimate ($)"].max()],
        palette=["lightblue", "green"],
    )
    .opt_table_font(font="Arial Narrow")
    .cols_align(align="left")
)

summary_gt

In [ ]:
from hdb_valuation.plots import plot_subject_vs_market

fig = plot_subject_vs_market(df_comp_recent, subject, blended_estimate, range_low, range_high)
plt.show()